In [99]:
from datetime import datetime, timedelta
from os import getenv

import numpy as np
import pandas as pd
from joblib import load

from utils import calculate_indicators_for_prediction_modul, get_candles, get_stocks

In [100]:
# загружаем модель
model = load('stocks_analyser_tickers_with_model.joblib')

In [101]:
# выгружаю токен для т-инвестиций
TOKEN = getenv("t_bank_token")
# P.S: если захотите воспользоваться этим модулем, вам придётся создать свой токен на сайте т-банка
# и записать его в эту переменную в таком формате: TOKEN = "ваш_токен"

In [102]:
# получаем данные по акциям
stocks = await get_stocks(TOKEN)

In [103]:
# создаем диапозон дат за 70 дней
end_date = datetime.now()
start_date = end_date - timedelta(days=70)

# выгружаем свечи в этом дипазоне
candles_df = await get_candles(TOKEN, stocks, start_date, end_date)
candles_df = candles_df.sort_values(['ticker', 'date']).reset_index(drop=True)

Запрашиваем данные с 2026-08-22 по 2026-10-31
Всего тикеров: 152


In [104]:
# фильтруем акции по обученному списку тикеров
#candles_df = candles_df[candles_df.ticker.isin(tickers)]
# проводим форматирование даты
candles_df.date = pd.to_datetime(candles_df.date)
# формируем признаки
candles_df = calculate_indicators_for_prediction_modul(candles_df)

In [105]:
# удаляем бесконечности и NaN
candles_df.replace([np.inf, -np.inf], np.nan, inplace=True)
candles_df.dropna(inplace=True)
# смотрим результат
candles_df

,ticker,sector,date,open,high,low,close,volume,month,return,...,sma_10,sma_40,macd,signal,hist,williams_r,volatility_20,volatility_60,momentum,ad_line
39,ABIO,health_care,2026-10-02,44.32,45.48,43.58,44.28,117013,10,-0.000903,...,44.640,44.86150,-0.121758,-0.065215,-0.056543,-66.137566,0.012049,0.014377,0.977483,-110142.408355
40,ABIO,health_care,2026-10-03,44.48,44.86,44.08,44.54,9324,10,0.005872,...,44.734,44.85300,-0.113823,-0.074937,-0.038887,-59.259259,0.011957,0.014225,0.987145,-108468.869893
41,ABIO,health_care,2026-10-04,44.54,44.68,44.28,44.68,5646,10,0.003143,...,44.702,44.84400,-0.095141,-0.078977,-0.016164,-55.555556,0.011983,0.014056,1.006306,-102822.869893
42,ABIO,health_care,2026-10-05,44.46,45.30,44.18,44.42,75834,10,-0.005819,...,44.626,44.87150,-0.100160,-0.083214,-0.016946,-62.433862,0.012035,0.013912,1.011384,-146156.584179
43,ABIO,health_care,2026-10-06,44.60,44.82,44.12,44.44,55519,10,0.000450,...,44.542,44.87050,-0.101356,-0.086842,-0.014514,-63.243243,0.010360,0.013745,1.002708,-150915.355607
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10179,ZAYM,financial,2026-10-27,126.65,128.45,126.30,127.25,11272,10,0.006725,...,126.420,127.38125,-0.709021,-0.911065,0.202044,-27.586207,0.007403,0.006940,1.009520,-51468.243004
10180,ZAYM,financial,2026-10-28,127.40,133.15,126.55,132.30,72533,10,0.039686,...,126.980,127.47125,-0.191237,-0.767099,0.575862,-10.059172,0.011623,0.008616,1.051669,2382.014572
10181,ZAYM,financial,2026-10-29,132.35,132.95,131.45,132.50,9224,10,0.001512,...,127.550,127.58625,0.232567,-0.567166,0.799733,-7.975460,0.011603,0.008423,1.054936,6071.614572
10182,ZAYM,financial,2026-10-30,132.70,132.90,131.25,131.80,6535,10,-0.005283,...,128.030,127.68375,0.506117,-0.352509,0.858626,-16.564417,0.011042,0.008388,1.049363,3893.281238


In [106]:
# выставляем выбраный при обучение порог
threshold = 0.66

In [107]:
# убираем колонки, которые нельзя использовать в обучении + те, которые мешали обучению
columns_to_drop =  ['index','date', 'target','return_7d','ticker','sector', 'close', 'low', 'open', 'high', 'return']

In [108]:
# формируем предсказания
pred = model.predict_proba(candles_df.drop(columns=columns_to_drop, errors='ignore'))[:,1] > threshold

In [109]:
candles_df[pred].sort_values('date')

,ticker,sector,date,open,high,low,close,volume,month,return,...,sma_10,sma_40,macd,signal,hist,williams_r,volatility_20,volatility_60,momentum,ad_line
8146,SMLT,real_estate,2026-10-02,229.60,237.00,227.40,230.60,6627605,10,0.004355,...,244.660,310.11000,-29.020075,-29.125105,0.105030,-84.924623,0.047072,0.039490,0.907160,-2.691153e+07
8147,SMLT,real_estate,2026-10-03,230.20,231.80,230.20,231.20,210047,10,0.002602,...,242.220,306.70500,-28.341295,-28.968343,0.627048,-83.248731,0.047212,0.039048,0.930008,-2.685902e+07
8148,SMLT,real_estate,2026-10-04,231.20,231.40,229.00,230.60,369797,10,-0.002595,...,240.040,303.38500,-27.534374,-28.681549,1.147176,-83.957219,0.046277,0.038582,0.958437,-2.673576e+07
8149,SMLT,real_estate,2026-10-05,230.40,234.00,221.40,230.40,8182303,10,-0.000867,...,237.920,300.18000,-26.604342,-28.266108,1.661766,-76.562500,0.034855,0.038143,0.990542,-2.322905e+07
4684,MGKL,financial,2026-10-26,1.87,1.89,1.53,1.56,1338743,10,-0.165775,...,1.888,1.93450,-0.046138,-0.018469,-0.027669,-94.000000,0.043005,0.028938,0.808290,-1.713186e+06
4685,MGKL,financial,2026-10-27,1.56,1.60,1.41,1.41,1614058,10,-0.096154,...,1.834,1.92025,-0.079102,-0.030596,-0.048506,-100.000000,0.047329,0.031332,0.730570,-3.327244e+06
4686,MGKL,financial,2026-10-28,1.36,1.44,1.27,1.27,1938782,10,-0.099291,...,1.766,1.90150,-0.115195,-0.047516,-0.067679,-100.000000,0.049330,0.033683,0.668421,-5.266026e+06
4687,MGKL,financial,2026-10-29,1.26,1.26,0.99,0.99,1561074,10,-0.220472,...,1.670,1.87700,-0.164496,-0.070912,-0.093584,-100.000000,0.070463,0.045768,0.526596,-6.827100e+06
4688,MGKL,financial,2026-10-30,0.89,0.89,0.89,0.89,169042,10,-0.101010,...,1.563,1.85000,-0.209225,-0.098574,-0.110651,-100.000000,0.071957,0.047335,0.475936,-6.827100e+06
4689,MGKL,financial,2026-10-31,0.86,0.86,0.86,0.86,91493,10,-0.033708,...,1.456,1.82225,-0.244278,-0.127715,-0.116563,-100.000000,0.071638,0.047299,0.551282,-6.827100e+06


In [110]:
# выводим предсказание на текущий день
today = candles_df[pred & (candles_df.date.dt.date == datetime.now().date())]
today

,ticker,sector,date,open,high,low,close,volume,month,return,...,sma_10,sma_40,macd,signal,hist,williams_r,volatility_20,volatility_60,momentum,ad_line
4689,MGKL,financial,2026-10-31,0.86,0.86,0.86,0.86,91493,10,-0.033708,...,1.456,1.82225,-0.244278,-0.127715,-0.116563,-100.0,0.071638,0.047299,0.551282,-6.827100e+06


In [ ]:
# predictions = pd.read_csv('data/predictions.csv')

# today = today[['date','ticker']]
# today['date'] = today['date'].dt.date
# new_df = pd.concat([predictions, today], ignore_index=True)

# new_df.to_csv('data//predictions.csv', index=False)

,date,ticker
0,2026-10-02,SMLT
1,2026-10-03,SMLT
2,2026-10-04,SMLT
3,2026-10-05,SMLT
4,2026-10-26,MGKL
5,2026-10-27,MGKL
6,2026-10-28,MGKL
7,2026-10-29,MGKL
8,2026-10-30,MGKL
9,2026-10-31,MGKL
